In [1]:
from pathlib import Path
import polars as pl

# Mencari folder dataset
candidates = [
    Path("../data/raw"),
    Path("data/raw"),
    Path("/home/jovyan/work/data/raw"),
]

data_dir = next(
    (
        p.resolve()
        for p in candidates
        if p.exists() and any(p.glob("*.parquet"))
    ),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan di folder data/raw"
    )

files = sorted(data_dir.glob("*.parquet"))

print("File dataset:")
for file in files:
    print(
        f"- {file.name}: "
        f"{file.stat().st_size / 1_000_000:.2f} MB"
    )

# Membaca dataset dengan lazy execution
df = pl.scan_parquet([str(file) for file in files])

print("\nStruktur kolom:")
print(df.collect_schema())

print("\nContoh 5 baris pertama:")
display(df.head(5).collect())

File dataset:
- mc4_id_0000.parquet: 289.05 MB
- mc4_id_0001.parquet: 291.06 MB

Struktur kolom:
Schema({'text': String, 'timestamp': String, 'url': String})

Contoh 5 baris pertama:


text,timestamp,url
str,str,str
"""Rentetan Program Kegiatan Prud…","""2020-01-23T09:19:49Z""","""https://www.prudential.co.id/i…"
"""TRISHCO (Trisakti Hospitality …","""2019-11-21T05:28:36Z""","""http://stptrisakti.ac.id/trish…"
"""Rapat Koordinasi Lintas Sektor…","""2020-02-22T23:02:46Z""","""https://wartanusa.id/rapat-koo…"
"""Nusa Penida Festival 2019, Kem…","""2020-06-02T18:13:32Z""","""http://metrobali.com/festival-…"
"""Wisata Buatan Archives - Dinas…","""2020-04-07T16:06:04Z""","""https://visitingjogja.com/info…"


In [2]:
# 1. Menghitung jumlah seluruh baris
jumlah_baris = df.select(
    pl.len().alias("jumlah_baris")
).collect()

print("Jumlah baris dataset:")
display(jumlah_baris)

# 2. Memeriksa jumlah nilai kosong setiap kolom
kolom = df.collect_schema().names()

missing_values = df.select([
    pl.col(c).null_count().alias(c)
    for c in kolom
]).collect()

print("Jumlah missing values setiap kolom:")
display(missing_values)

Jumlah baris dataset:


jumlah_baris
u32
305000


Jumlah missing values setiap kolom:


text,timestamp,url
u32,u32,u32
0,0,0


In [3]:
# Profil tambahan untuk memeriksa kualitas data
profil_lanjutan = df.select(
    pl.col("text")
      .str.len_chars()
      .mean()
      .round(2)
      .alias("rata_rata_panjang_teks"),

    pl.col("text")
      .str.len_chars()
      .min()
      .alias("panjang_teks_minimum"),

    pl.col("text")
      .str.len_chars()
      .max()
      .alias("panjang_teks_maksimum"),

    (pl.col("text").str.strip_chars() == "")
      .sum()
      .alias("jumlah_teks_kosong"),

    pl.col("url")
      .n_unique()
      .alias("jumlah_url_unik"),

    pl.col("timestamp")
      .min()
      .alias("timestamp_minimum"),

    pl.col("timestamp")
      .max()
      .alias("timestamp_maksimum"),
).collect()

display(profil_lanjutan)

rata_rata_panjang_teks,panjang_teks_minimum,panjang_teks_maksimum,jumlah_teks_kosong,jumlah_url_unik,timestamp_minimum,timestamp_maksimum
f64,u32,u32,u32,u32,str,str
3160.02,62,185713,0,305000,"""2013-05-18T05:01:28Z""","""2020-08-15T21:24:00Z"""


In [4]:
# Memeriksa rentang waktu dan keunikan teks
ringkasan_data = df.select(
    pl.col("timestamp").min().alias("timestamp_awal"),
    pl.col("timestamp").max().alias("timestamp_akhir"),
    pl.col("text").n_unique().alias("jumlah_teks_unik"),
).collect()

print(ringkasan_data.to_dicts())

[{'timestamp_awal': '2013-05-18T05:01:28Z', 'timestamp_akhir': '2020-08-15T21:24:00Z', 'jumlah_teks_unik': 305000}]
